In [6]:
# Cell 1: imports
import math
from dataclasses import dataclass
import os

import numpy as np
import warp as wp
from tqdm import tqdm


import newton
import newton.examples
import newton.viewer
import trimesh

In [2]:
def add_cylindrical_container(
    builder,
    radius,
    height,
    static_cfg,
    wall_thickness=0.3,
    floor_thickness=0.25,
    n_faces=32,
    center=(0.0, 0.0, 0.0),
):
    """
    Create an open-top cylindrical container approximated by box wall segments.

    Newton convention: x, y, z

    Args:
        builder: Newton ModelBuilder
        radius (float): inner radius of the container
        height (float): wall height along Z
        static_cfg: ShapeConfig for static objects
        wall_thickness (float): radial wall thickness
        floor_thickness (float): floor thickness
        n_faces (int): number of wall segments
        center (tuple): (x, y, z) centre of the container base
    """

    if n_faces < 3:
        raise ValueError("n_faces must be at least 3")

    cx, cy, cz = center
    static_body = -1

    # -----------------------------
    # Floor
    # -----------------------------
    builder.add_shape_cylinder(
        static_body,
        radius=radius,
        half_height=floor_thickness * 0.5,
        xform=wp.transform(
            p=wp.vec3(cx, cy, cz + floor_thickness * 0.5),
            q=wp.quat_identity(),
        ),
        cfg=static_cfg,
    )

    # -----------------------------
    # Wall segments
    # -----------------------------
    angle_step = 2.0 * math.pi / n_faces

    # Segment chord length around the inner radius
    segment_length = 2.0 * radius * math.tan(math.pi / n_faces)

    for i in range(n_faces):
        theta = i * angle_step

        # Wall centre is halfway through the wall thickness
        wall_radius = radius + wall_thickness * 0.5

        x = cx + wall_radius * math.cos(theta)
        y = cy + wall_radius * math.sin(theta)

        # Rotate each box so its long side is tangent to the circle
        q = wp.quat_from_axis_angle(
            wp.vec3(0.0, 0.0, 1.0),
            theta,
        )

        builder.add_shape_box(
            static_body,
            hx=wall_thickness * 0.5,
            hy=segment_length * 0.5,
            hz=height * 0.5,
            xform=wp.transform(
                p=wp.vec3(x, y, cz + height * 0.5),
                q=q,
            ),
            cfg=static_cfg,
        )

def add_particle(
    builder,
    path,
    position,
    orientation,
    scale=1.0,
    dynamic_cfg=None,
):
    """
    Add an OBJ particle mesh to a Newton ModelBuilder.

    Collision uses the convex hull of the OBJ mesh.

    Parameters
    ----------
    builder : newton.ModelBuilder
    path : str
        Path to .obj file.
    position : tuple/list/wp.vec3
        Newton uses (x, y, z).
    orientation : tuple/list/wp.quat
        Quaternion as (x, y, z, w), or wp.quat.
    scale : float
    dynamic_cfg : newton.ModelBuilder.ShapeConfig | None
        If None, a dynamic config is created.

    Returns
    -------
    body_id : int
    """

    # -----------------------------
    # Load OBJ mesh
    # -----------------------------
    mesh = trimesh.load(path, force="mesh")

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Expected a trimesh.Trimesh, got {type(mesh)}")

    # Apply scale
    mesh = mesh.copy()
    vertices = mesh.vertices
    com = vertices.mean(axis=0)
    vertices_centered = vertices - com
    mesh.vertices = vertices_centered
    mesh.apply_scale(scale)

    # -----------------------------
    # Use convex hull for collision
    # -----------------------------
    hull = mesh.convex_hull

    vertices = np.asarray(hull.vertices, dtype=np.float32)
    faces = np.asarray(hull.faces, dtype=np.int32)

    if vertices.shape[0] == 0 or faces.shape[0] == 0:
        raise ValueError("Convex hull mesh is empty.")

    # Newton expects flattened triangle indices
    indices = faces.reshape(-1).astype(np.int32)

    collision_mesh = newton.Mesh(
        vertices,
        indices,
        compute_inertia=True,
        is_solid=True,
    )

    # -----------------------------
    # Shape config
    # -----------------------------
    if dynamic_cfg is None:
        cfg = newton.ModelBuilder.ShapeConfig()
        cfg.density = 1000.0
        cfg.ke = 1.0e5
        cfg.kd = 1.0e-4
        cfg.mu = 0.6
        cfg.restitution = 0.35
    else:
        cfg = dynamic_cfg

    # -----------------------------
    # Transform
    # -----------------------------
    if isinstance(position, wp.vec3):
        p = position
    else:
        p = wp.vec3(float(position[0]), float(position[1]), float(position[2]))

    if isinstance(orientation, wp.quat):
        q = orientation
    else:
        q = wp.quat(
            float(orientation[0]),
            float(orientation[1]),
            float(orientation[2]),
            float(orientation[3]),
        )

    # -----------------------------
    # Dynamic body
    # -----------------------------
    body_id = builder.add_body(
        xform=wp.transform(p=p, q=q),
        label="particle",
    )

    builder.add_shape_mesh(
        body=body_id,
        mesh=collision_mesh,
        cfg=cfg,
    )

    return body_id

In [3]:
# -----------------------------
# Viewer
# -----------------------------
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)

# -----------------------------
# Simulation settings
# -----------------------------
frame_dt = 1.0 / 60.0
sim_substeps = 10
sim_dt = frame_dt / sim_substeps
sim_time = 0.0
num_frames = 600

# -----------------------------
# Build scene
# -----------------------------
builder = newton.ModelBuilder()

# -----------------------------
# Materials
# -----------------------------
static_cfg = newton.ModelBuilder.ShapeConfig()
static_cfg.ke = 1.0e5
static_cfg.kd = 1.0e-4
static_cfg.mu = 0.8
static_cfg.restitution = 0.2

dynamic_cfg = newton.ModelBuilder.ShapeConfig()
dynamic_cfg.density = 1000.0
dynamic_cfg.ke = 1.0e5
dynamic_cfg.kd = 1.0e-4
dynamic_cfg.mu = 0.6
dynamic_cfg.restitution = 0.35

# -----------------------------
# Open square box without lid
# -----------------------------
# Newton convention: x, y, z
box_half_width = 4.0
wall_thickness = 0.3
wall_height = 4.0
floor_thickness = 0.25

# Static world body is usually -1
static_body = -1

add_cylindrical_container(
    builder,
    radius=4.0,
    height=4.0,
    static_cfg=static_cfg,
)
# # Floor
# builder.add_shape_box(
#     static_body,
#     hx=box_half_width,
#     hy=box_half_width,
#     hz=floor_thickness * 0.5,
#     xform=wp.transform(
#         p=wp.vec3(0.0, 0.0, floor_thickness * 0.5),
#         q=wp.quat_identity(),
#     ),
#     cfg=dynamic_cfg,
# )

# # +X wall
# builder.add_shape_box(
#     static_body,
#     hx=wall_thickness * 0.5,
#     hy=box_half_width,
#     hz=wall_height * 0.5,
#     xform=wp.transform(
#         p=wp.vec3(box_half_width + wall_thickness * 0.5, 0.0, wall_height * 0.5),
#         q=wp.quat_identity(),
#     ),
#     cfg=dynamic_cfg,
# )

# # -X wall
# builder.add_shape_box(
#     static_body,
#     hx=wall_thickness * 0.5,
#     hy=box_half_width,
#     hz=wall_height * 0.5,
#     xform=wp.transform(
#         p=wp.vec3(-box_half_width - wall_thickness * 0.5, 0.0, wall_height * 0.5),
#         q=wp.quat_identity(),
#     ),
#     cfg=dynamic_cfg,
# )

# # +Y wall
# builder.add_shape_box(
#     static_body,
#     hx=box_half_width,
#     hy=wall_thickness * 0.5,
#     hz=wall_height * 0.5,
#     xform=wp.transform(
#         p=wp.vec3(0.0, box_half_width + wall_thickness * 0.5, wall_height * 0.5),
#         q=wp.quat_identity(),
#     ),
#     cfg=dynamic_cfg,
# )

# # -Y wall
# builder.add_shape_box(
#     static_body,
#     hx=box_half_width,
#     hy=wall_thickness * 0.5,
#     hz=wall_height * 0.5,
#     xform=wp.transform(
#         p=wp.vec3(0.0, -box_half_width - wall_thickness * 0.5, wall_height * 0.5),
#         q=wp.quat_identity(),
#     ),
#     cfg=dynamic_cfg,
# )

# -----------------------------
# Random rotations
# -----------------------------
rng = np.random.default_rng(42)

def random_quat():
    axis = rng.normal(size=3)
    axis = axis / np.linalg.norm(axis)
    angle = float(rng.uniform(0.0, 2.0 * math.pi))
    return wp.quat_from_axis_angle(
        wp.vec3(float(axis[0]), float(axis[1]), float(axis[2])),
        angle,
    )

# -----------------------------
# Five shapes poured from z=10 to z=30
# -----------------------------
# z_positions = np.linspace(10.0, 30.0, 5)

# # Put them above the box opening with slight XY offsets
# xy_positions = [
#     (-0.8, -0.6),
#     ( 0.7, -0.4),
#     ( 0.0,  0.5),
#     (-0.5,  0.7),
#     ( 0.6,  0.6),
# ]

# # Sphere
# body = builder.add_body(
#     xform=wp.transform(
#         p=wp.vec3(xy_positions[0][0], xy_positions[0][1], float(z_positions[0])),
#         q=random_quat(),
#     ),
#     label="sphere",
# )
# builder.add_shape_sphere(body, radius=0.5, cfg=dynamic_cfg)

# # Box
# body = builder.add_body(
#     xform=wp.transform(
#         p=wp.vec3(xy_positions[1][0], xy_positions[1][1], float(z_positions[1])),
#         q=random_quat(),
#     ),
#     label="box",
# )
# builder.add_shape_box(body, hx=0.5, hy=0.3, hz=0.8, cfg=dynamic_cfg)

# # Cone
# body = builder.add_body(
#     xform=wp.transform(
#         p=wp.vec3(xy_positions[2][0], xy_positions[2][1], float(z_positions[2])),
#         q=random_quat(),
#     ),
#     label="cone",
# )
# builder.add_shape_cone(body, radius=0.4, half_height=0.6, cfg=dynamic_cfg)

# # Cylinder
# body = builder.add_body(
#     xform=wp.transform(
#         p=wp.vec3(xy_positions[3][0], xy_positions[3][1], float(z_positions[3])),
#         q=random_quat(),
#     ),
#     label="cylinder",
# )
# builder.add_shape_cylinder(body, radius=0.35, half_height=0.5, cfg=dynamic_cfg)

# # Capsule
# body = builder.add_body(
#     xform=wp.transform(
#         p=wp.vec3(xy_positions[4][0], xy_positions[4][1], float(z_positions[4])),
#         q=random_quat(),
#     ),
#     label="capsule",
# )
# builder.add_shape_capsule(body, radius=0.3, half_height=0.5, cfg=dynamic_cfg)

add_particle(
    builder,
    '/home/bg/Developer/particle-pack-generation/data/toy_data_GOH_6_250/tmp/1944.obj',
    position=(0.0, 0.0, 40.0),
    orientation=random_quat(),
    scale=0.01,
    dynamic_cfg=dynamic_cfg
)

# -----------------------------
# Finalise model
# -----------------------------
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(
    model,
    broad_phase="sap",
)

solver = newton.solvers.SolverXPBD(
    model,
    iterations=12,
    rigid_contact_relaxation=0.8,
)

state_0 = model.state()
state_1 = model.state()
control = model.control()

newton.eval_fk(
    model,
    model.joint_q,
    model.joint_qd,
    state_0,
)

contacts = collision_pipeline.contacts()

# -----------------------------
# Viewer setup
# -----------------------------
viewer.set_model(model)

viewer.set_camera(
    pos=wp.vec3(10.0, -14.0, 10.0),
    pitch=-30.0,
    yaw=135.0,
)



Module newton._src.geometry.inertia 5855353 load on device 'cuda:0' took 0.72 ms  (cached)
Module validate_and_correct_inertia_kernel_7693acc3 7693acc load on device 'cuda:0' took 0.28 ms  (cached)
Module newton._src.solvers.solver 1843e22 load on device 'cuda:0' took 0.59 ms  (cached)
Module newton._src.sim.articulation 3d2eb66 load on device 'cuda:0' took 3.83 ms  (cached)
Module newton._src.utils.mesh ccd157d load on device 'cuda:0' took 0.39 ms  (cached)
Module newton._src.viewer.kernels 9252450 load on device 'cuda:0' took 1.11 ms  (cached)


In [4]:
# -----------------------------
# Simulation loop
# -----------------------------
for frame in tqdm(range(num_frames)):
    for _ in range(sim_substeps):
        state_0.clear_forces()

        contacts = model.collide(
            state_0,
            collision_pipeline=collision_pipeline,
        )

        viewer.apply_forces(state_0)

        solver.step(
            state_0,
            state_1,
            control,
            contacts,
            sim_dt,
        )

        state_0, state_1 = state_1, state_0

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    sim_time += frame_dt

  0%|          | 1/600 [00:00<02:04,  4.79it/s]

Module newton._src.sim.collide 03e1931 load on device 'cuda:0' took 1.08 ms  (cached)
Module newton._src.geometry.broad_phase_sap cf68639 load on device 'cuda:0' took 0.47 ms  (cached)
Module create_narrow_phase_primitive_kernel__locals__narrow_phase_primitive_kernel_2fdc6afc 807ed5e load on device 'cuda:0' took 0.50 ms  (cached)
Module create_narrow_phase_kernel_gjk_mpr__locals__narrow_phase_kernel_gjk_mpr_ab64b611 e27c8a3 load on device 'cuda:0' took 1.51 ms  (cached)
Module newton._src.geometry.narrow_phase 854d9dc load on device 'cuda:0' took 0.54 ms  (cached)
Module newton._src.geometry.contact_reduction_global ce266e9 load on device 'cuda:0' took 0.30 ms  (cached)
Module newton._src.geometry.narrow_phase 8f09feb load on device 'cuda:0' took 0.60 ms  (cached)
Module newton._src.geometry.sdf_contact 819d988 load on device 'cuda:0' took 0.22 ms  (cached)
Module create_narrow_phase_process_mesh_plane_contacts_kernel__locals__narrow_phase_process_mesh_plane_contacts_reduce_kernel_03ed

100%|██████████| 600/600 [00:09<00:00, 62.95it/s]


In [5]:
viewer.show_notebook()

HTML(value='<div id="3880c5c4-91bb-4565-bc4e-b6d1ee187587"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

In [ ]:
@dataclass
class SimulationSettings:
    frame_dt: float = 1.0 / 60.0
    sim_substeps: int = 10
    sim_time: float = 0.0
    num_frames: int = 600

    def __post_init__(self):
        self.sim_dt = self.frame_dt / self.sim_substeps

sim_settings = SimulationSettings()

In [ ]:
class Simulator:
    def __init__(
        self,
        fps: int=60,
        sim_substeps: int=10,
        viewer = None,
        builder = None,
    ):
        self.fps = fps
        self.frame_dt = 1.0 / fps
        self.sim_substeps = sim_substeps
        self.sim_dt = self.frame_dt / sim_substeps

        self.viewer = viewer
        if builder is not None:
            self.builder = builder
        else:
            self.builder = newton.ModelBuilder()
    
    def finalise_model(self):
        self.model = self.builder.finalize()
        if self.viewer is not None:
            self.viewer.set_model(self.model)

    def run(self, num_frames):
        collision_pipeline = newton.CollisionPipeline(
            self.model,
            broad_phase="sap",
        )
        solver = newton.solvers.SolverXPBD(
            self.model,
            iterations=12,
            rigid_contact_relaxation=0.8,
        )
        state_0 = self.model.state()
        state_1 = self.model.state()
        control = self.model.control()
        newton.eval_fk(
            self.model,
            self.model.joint_q,
            self.model.joint_qd,
            state_0,
        )
        contacts = collision_pipeline.contacts()

        sim_time = 0.0
        for frame in tqdm(range(num_frames)):
            for _ in range(self.sim_substeps):
                state_0.clear_forces()
                contacts = self.model.collide(
                    state_0,
                    collision_pipeline=collision_pipeline,
                )
                if self.viewer is not None:
                    self.viewer.apply_forces(state_0)
                solver.step(
                    state_0,
                    state_1,
                    control,
                    contacts,
                    self.sim_dt,
                )
                state_0, state_1 = state_1, state_0

            if self.viewer is not None:
                self.viewer.begin_frame(sim_time)
                self.viewer.log_state(state_0)
                self.viewer.log_contacts(contacts, state_0)
                self.viewer.end_frame()
            sim_time += self.frame_dt
        
        return state_0

    def add_cylindrical_container(
        self,
        radius,
        height,
        static_cfg,
        wall_thickness=0.3,
        floor_thickness=0.25,
        n_faces=32,
        center=(0.0, 0.0, 0.0),
    ):
        """
        Create an open-top cylindrical container approximated by box wall segments.

        Newton convention: x, y, z

        Args:
            builder: Newton ModelBuilder
            radius (float): inner radius of the container
            height (float): wall height along Z
            static_cfg: ShapeConfig for static objects
            wall_thickness (float): radial wall thickness
            floor_thickness (float): floor thickness
            n_faces (int): number of wall segments
            center (tuple): (x, y, z) centre of the container base
        """
        if n_faces < 3:
            raise ValueError("n_faces must be at least 3")
        cx, cy, cz = center
        static_body = -1
        # -----------------------------
        # Floor
        # -----------------------------
        self.builder.add_shape_cylinder(
            static_body,
            radius=radius,
            half_height=floor_thickness * 0.5,
            xform=wp.transform(
                p=wp.vec3(cx, cy, cz + floor_thickness * 0.5),
                q=wp.quat_identity(),
            ),
            cfg=static_cfg,
        )
        # -----------------------------
        # Wall segments
        # -----------------------------
        angle_step = 2.0 * math.pi / n_faces
        # Segment chord length around the inner radius
        segment_length = 2.0 * radius * math.tan(math.pi / n_faces)
        for i in range(n_faces):
            theta = i * angle_step
            # Wall centre is halfway through the wall thickness
            wall_radius = radius + wall_thickness * 0.5
            x = cx + wall_radius * math.cos(theta)
            y = cy + wall_radius * math.sin(theta)
            # Rotate each box so its long side is tangent to the circle
            q = wp.quat_from_axis_angle(
                wp.vec3(0.0, 0.0, 1.0),
                theta,
            )
            self.builder.add_shape_box(
                static_body,
                hx=wall_thickness * 0.5,
                hy=segment_length * 0.5,
                hz=height * 0.5,
                xform=wp.transform(
                    p=wp.vec3(x, y, cz + height * 0.5),
                    q=q,
                ),
                cfg=static_cfg,
            )

    def add_particle(
        self,
        dir,
        id,
        position,
        orientation,
        scale=1.0,
        dynamic_cfg=None,
    ):
        """
        Add an OBJ particle mesh to a Newton ModelBuilder.

        Collision uses the convex hull of the OBJ mesh.

        Parameters
        ----------
        dir : str
            Directory containing the .obj file.
        id : int
            Unique identifier for the particle.
        position : tuple/list/wp.vec3
            Newton uses (x, y, z).
        orientation : tuple/list/wp.quat
            Quaternion as (x, y, z, w), or wp.quat.
        scale : float
        dynamic_cfg : newton.ModelBuilder.ShapeConfig | None
            If None, a dynamic config is created.

        Returns
        -------
        body_id : int
        """
        # -----------------------------
        # Load OBJ mesh
        # -----------------------------
        path = os.path.join(dir, f"{id}.obj")
        mesh = trimesh.load(path, force="mesh")
        if not isinstance(mesh, trimesh.Trimesh):
            raise TypeError(f"Expected a trimesh.Trimesh, got {type(mesh)}")
        # Apply scale
        mesh = mesh.copy()
        vertices = mesh.vertices
        com = vertices.mean(axis=0)
        vertices_centered = vertices - com
        mesh.vertices = vertices_centered
        mesh.apply_scale(scale)
        # -----------------------------
        # Use convex hull for collision
        # -----------------------------
        hull = mesh.convex_hull
        vertices = np.asarray(hull.vertices, dtype=np.float32)
        faces = np.asarray(hull.faces, dtype=np.int32)
        if vertices.shape[0] == 0 or faces.shape[0] == 0:
            raise ValueError("Convex hull mesh is empty.")
        # Newton expects flattened triangle indices
        indices = faces.reshape(-1).astype(np.int32)
        collision_mesh = newton.Mesh(
            vertices,
            indices,
            compute_inertia=True,
            is_solid=True,
        )
        # -----------------------------
        # Shape config
        # -----------------------------
        if dynamic_cfg is None:
            cfg = newton.ModelBuilder.ShapeConfig()
            cfg.density = 1000.0
            cfg.ke = 1.0e5
            cfg.kd = 1.0e-4
            cfg.mu = 0.6
            cfg.restitution = 0.35
        else:
            cfg = dynamic_cfg
        # -----------------------------
        # Transform
        # -----------------------------
        if isinstance(position, wp.vec3):
            p = position
        else:
            p = wp.vec3(float(position[0]), float(position[1]), float(position[2]))
        if isinstance(orientation, wp.quat):
            q = orientation
        else:
            q = wp.quat(
                float(orientation[0]),
                float(orientation[1]),
                float(orientation[2]),
                float(orientation[3]),
            )
        # -----------------------------
        # Dynamic body
        # -----------------------------
        body_id = self.builder.add_body(
            xform=wp.transform(p=p, q=q),
            label=f"particle_{id}",
        )
        self.builder.add_shape_mesh(
            body=body_id,
            mesh=collision_mesh,
            cfg=cfg,
        )
        return body_id

In [44]:
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
simulator = Simulator(
    fps=60,
    sim_substeps=10,
    viewer=viewer,
)

In [ ]:
simulator.add_cylindrical_container(
    radius=4.0,
    height=4.0,
    static_cfg=newton.ModelBuilder.ShapeConfig(
        ke=1.0e5,
        kd=1.0e-4,
        mu=0.8,
        restitution=0.2,
    ),
)
body_ids = {}

for idx, i in enumerate([702, 821, 1944]):
    body_ids[i] = simulator.add_particle(
        dir='/home/bg/Developer/particle-pack-generation/data/toy_data_GOH_6_250/tmp',
        id=i,
        position=(0.0, 0.0, 40.0 + idx * 10.0),
        orientation=(0, 0, 0, 1),
        scale=0.01,
        dynamic_cfg=newton.ModelBuilder.ShapeConfig(
            density=1000.0,
            ke=1.0e5,
            kd=1.0e-4,
            mu=0.6,
            restitution=0.35,
        ),
    )

simulator.finalise_model()

In [46]:
state = simulator.run(num_frames=600)

100%|██████████| 600/600 [00:09<00:00, 62.70it/s]


In [47]:
viewer.show_notebook()

HTML(value='<div id="d24cc397-c4ae-4759-9dae-09498ca8a6bd"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

In [53]:
state.body_q.numpy()

array([[-1.8367791 , -1.4093121 ,  1.1517463 ,  0.12890051, -0.6952448 ,
         0.12890625,  0.6952716 ],
       [ 1.3857536 ,  2.5619986 ,  0.59090924,  0.47377512,  0.3567676 ,
        -0.0895333 ,  0.8001486 ],
       [ 1.1355875 , -1.8530309 ,  0.9168756 ,  0.9768126 ,  0.0651243 ,
        -0.20172685,  0.03003624]], dtype=float32)

In [35]:
print(body_ids)

{702: 0, 821: 1, 1944: 2}
